In [1]:
import os
import random
import numpy as np
import soundfile as sf
import librosa

In [ ]:
# ================================
# 경로 설정
# ================================
INPUT_DIR = '/home/isol/work/data/raw/jurion_speech'   # 원본 호리야
NOISE_DIR    = '/home/isol/work/data/train/noise_speech_cut'   # 노이즈 파일들
OUTPUT_DIR   = '/home/isol/work/data/raw/jurion_speech_noisy'  # 증강 결과 저장

AI_HUB_WAV_DIR   = '/home/isol/work/data/train/AI_Hub/wav'
AI_HUB_NOISY_DIR = '/home/isol/work/data/train/AI_Hub/wav_speech_noise'

SAMPLE_RATE  = 16000
SAMPLE_PER_FOLDER = 5000
SEED         = 42

# SNR 범위 (dB) - 낮을수록 노이즈가 강함
SNR_MIN = 1   # 노이즈가 꽤 강함
SNR_MAX = 5  # 노이즈가 약함

random.seed(SEED)
np.random.seed(SEED)

In [5]:
def load_audio(path, sr=SAMPLE_RATE):
    """오디오 로드 후 모노로 변환"""
    audio, orig_sr = librosa.load(path, sr=sr, mono=True)
    return audio

def get_noise_segment(noise_audio, target_len):
    """노이즈 파일에서 랜덤 구간 추출"""
    noise_len = len(noise_audio)

    if noise_len >= target_len:
        # 노이즈가 충분히 길면 랜덤 구간 추출
        start = random.randint(0, noise_len - target_len)
        return noise_audio[start:start + target_len]
    else:
        # 노이즈가 짧으면 반복해서 채움
        repeats = (target_len // noise_len) + 1
        noise_tiled = np.tile(noise_audio, repeats)
        start = random.randint(0, len(noise_tiled) - target_len)
        return noise_tiled[start:start + target_len]
    
def mix_with_noise(signal, noise, snr_db):
    """
    SNR(Signal-to-Noise Ratio)을 맞춰서 노이즈 믹싱
    snr_db가 높을수록 노이즈가 적음
    """
    # 신호와 노이즈 파워 계산
    signal_power = np.mean(signal ** 2)
    noise_power = np.mean(noise ** 2)

    if noise_power == 0:
        return signal

    # SNR에 맞게 노이즈 스케일 조정
    snr_linear = 10 ** (snr_db / 10)
    scale = np.sqrt(signal_power / (snr_linear * noise_power + 1e-9))
    noisy_signal = signal + scale * noise

    # 클리핑 방지
    max_val = np.max(np.abs(noisy_signal))
    if max_val > 1.0:
        noisy_signal = noisy_signal / max_val * 0.95

    return noisy_signal

def augment_wakeword():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 호리야 파일 목록
    wakeword_files = [
        f for f in os.listdir(INPUT_DIR)
        if f.lower().endswith('.wav')
    ]

    # 노이즈 파일 목록
    noise_files = [
        f for f in os.listdir(NOISE_DIR)
        if f.lower().endswith('.wav')
    ]

    print(f"인풋 원본: {len(wakeword_files)}개")
    print(f"노이즈 파일: {len(noise_files)}개")
    print(f"저장 위치: {OUTPUT_DIR}")
    print(f"SNR 범위: {SNR_MIN}~{SNR_MAX} dB")
    print("="*40)

    # 노이즈 파일 미리 로드 (매번 로드하면 느리니까)
    print("노이즈 파일 로드 중...")
    noise_audios = []
    for nf in noise_files:
        noise_path = os.path.join(NOISE_DIR, nf)
        try:
            noise = load_audio(noise_path)
            noise_audios.append(noise)
        except Exception as e:
            print(f"  [WARNING] {nf} 로드 실패: {e}")
    print(f"노이즈 {len(noise_audios)}개 로드 완료!\n")

    success = 0
    fail    = 0

    for i, wf in enumerate(wakeword_files):
        wav_path = os.path.join(INPUT_DIR, wf)

        try:
            # 원본 로드
            signal = load_audio(wav_path)

            # 랜덤 노이즈 선택
            noise_audio = random.choice(noise_audios)

            # 노이즈 구간 추출 (원본과 같은 길이)
            noise_segment = get_noise_segment(noise_audio, len(signal))

            # 랜덤 SNR 선택
            snr_db = random.uniform(SNR_MIN, SNR_MAX)

            # 믹싱
            noisy_signal = mix_with_noise(signal, noise_segment, snr_db)

            # 저장 (파일명에 _noisy 추가)
            base_name = os.path.splitext(wf)[0]
            output_path = os.path.join(OUTPUT_DIR, f"{base_name}_noisy.wav")
            sf.write(output_path, noisy_signal, SAMPLE_RATE)

            success += 1
            if (i + 1) % 500 == 0:
                print(f"  {i+1}/{len(wakeword_files)} 완료...")

        except Exception as e:
            print(f"  [ERROR] {wf}: {e}")
            fail += 1

    print(f"\n{'='*40}")
    print(f"증강 완료!")
    print(f"  성공: {success}개")
    print(f"  실패: {fail}개")
    print(f"  저장 위치: {OUTPUT_DIR}")
    print(f"\n원본 {len(wakeword_files)}개 + 증강 {success}개 = 총 {len(wakeword_files)+success}개")

def augment_aihub():
    # 노이즈 파일 미리 로드
    print("노이즈 파일 로드 중...")
    noise_files = [
        f for f in os.listdir(NOISE_DIR)
        if f.lower().endswith('.wav')
    ]
    noise_audios = []
    for nf in noise_files:
        try:
            noise = load_audio(os.path.join(NOISE_DIR, nf))
            noise_audios.append(noise)
        except Exception as e:
            print(f" [WARNING] {nf}: {e}")
    print(f"노이즈 {len(noise_audios)}개 로드 완료!\n")

    # 단어 폴더 목록
    word_folders = sorted([
        d for d in os.listdir(AI_HUB_WAV_DIR)
        if os.path.isdir(os.path.join(AI_HUB_WAV_DIR, d))
    ])
    print(f"단어 폴더 수: {len(word_folders)}개")
    print(f"폴더당 샘플: {SAMPLE_PER_FOLDER}개")
    print(f"예상 총 출력: {len(word_folders) * SAMPLE_PER_FOLDER}개")
    print("="*40)

    total_success = 0
    total_fail    = 0

    for word in word_folders:
        src_folder = os.path.join(AI_HUB_WAV_DIR, word)
        dst_folder = os.path.join(AI_HUB_NOISY_DIR, word)
        os.makedirs(dst_folder, exist_ok=True)

        # wav 파일 목록에서 100개 랜덤 선택
        wav_files = [
            f for f in os.listdir(src_folder)
            if f.lower().endswith('.wav')
        ]
        sample_size = min(SAMPLE_PER_FOLDER, len(wav_files))
        selected_wavs = random.sample(wav_files, sample_size)

        success = 0
        fail    = 0

        for wf in selected_wavs:
            src_path = os.path.join(src_folder, wf)
            try:
                signal          = load_audio(src_path)
                noise_audio     = random.choice(noise_audios)
                noise_segment   = get_noise_segment(noise_audio, len(signal))
                snr_db          = random.uniform(SNR_MIN, SNR_MAX)
                noisy_signal    = mix_with_noise(signal, noise_segment, snr_db)

                base_name  = os.path.splitext(wf)[0]
                dst_path   = os.path.join(dst_folder, f"{base_name}_noisy.wav")
                sf.write(dst_path, noisy_signal, SAMPLE_RATE)
                success += 1
            
            except Exception as e:
                print(f"  [ERROR] {word}/{wf}: {e}")
                fail += 1
        
        total_success += success
        total_fail    += fail
        print(f"  {word:15s}: {success}개 완료")

    print(f"\n{'='*40}")
    print(f"전체 완료!")
    print(f"  성공: {total_success}개")
    print(f"  실패: {total_fail}개")
    print(f"  저장 위치: {AI_HUB_NOISY_DIR}")

In [43]:
INPUT_DIR = '/home/isol/work/data/train/wakeword_clean'   # 원본 호리야
NOISE_DIR    = '/home/isol/work/data/train/noise_bg_cut'   # 노이즈 파일들
OUTPUT_DIR   = '/home/isol/work/data/train/wakeword_bg_noisy_up'  # 증강 결과 저장

augment_wakeword()

인풋 원본: 7148개
노이즈 파일: 21868개
저장 위치: /home/isol/work/data/train/wakeword_bg_noisy_up
SNR 범위: 1~5 dB
노이즈 파일 로드 중...
노이즈 21868개 로드 완료!

  500/7148 완료...
  1000/7148 완료...
  1500/7148 완료...
  2000/7148 완료...
  2500/7148 완료...
  3000/7148 완료...
  3500/7148 완료...
  4000/7148 완료...
  4500/7148 완료...
  5000/7148 완료...
  5500/7148 완료...
  6000/7148 완료...
  6500/7148 완료...
  7000/7148 완료...

증강 완료!
  성공: 7148개
  실패: 0개
  저장 위치: /home/isol/work/data/train/wakeword_bg_noisy_up

원본 7148개 + 증강 7148개 = 총 14296개


In [44]:
INPUT_DIR = '/home/isol/work/data/train/wakeword_clean'   # 원본 호리야
NOISE_DIR    = '/home/isol/work/data/train/noise_speech_cut'   # 노이즈 파일들
OUTPUT_DIR   = '/home/isol/work/data/train/wakeword_speech_noisy_up'  # 증강 결과 저장

augment_wakeword()

인풋 원본: 7148개
노이즈 파일: 32079개
저장 위치: /home/isol/work/data/train/wakeword_speech_noisy_up
SNR 범위: 1~5 dB
노이즈 파일 로드 중...
노이즈 32079개 로드 완료!

  500/7148 완료...
  1000/7148 완료...
  1500/7148 완료...
  2000/7148 완료...
  2500/7148 완료...
  3000/7148 완료...
  3500/7148 완료...
  4000/7148 완료...
  4500/7148 완료...
  5000/7148 완료...
  5500/7148 완료...
  6000/7148 완료...
  6500/7148 완료...
  7000/7148 완료...

증강 완료!
  성공: 7148개
  실패: 0개
  저장 위치: /home/isol/work/data/train/wakeword_speech_noisy_up

원본 7148개 + 증강 7148개 = 총 14296개


In [45]:
INPUT_DIR = '/home/isol/work/data/train/wakeword_clean'   # 원본 호리야
NOISE_DIR    = '/home/isol/work/data/train/noise_both_cut'   # 노이즈 파일들
OUTPUT_DIR   = '/home/isol/work/data/train/wakeword_both_noisy_up'  # 증강 결과 저장

augment_wakeword()

인풋 원본: 7148개
노이즈 파일: 81891개
저장 위치: /home/isol/work/data/train/wakeword_both_noisy_up
SNR 범위: 1~5 dB
노이즈 파일 로드 중...
노이즈 81891개 로드 완료!

  500/7148 완료...
  1000/7148 완료...
  1500/7148 완료...
  2000/7148 완료...
  2500/7148 완료...
  3000/7148 완료...
  3500/7148 완료...
  4000/7148 완료...
  4500/7148 완료...
  5000/7148 완료...
  5500/7148 완료...
  6000/7148 완료...
  6500/7148 완료...
  7000/7148 완료...

증강 완료!
  성공: 7148개
  실패: 0개
  저장 위치: /home/isol/work/data/train/wakeword_both_noisy_up

원본 7148개 + 증강 7148개 = 총 14296개


In [46]:
INPUT_DIR = '/home/isol/work/data/train/wakeword_clean'   # 원본 호리야
NOISE_DIR    = '/home/isol/work/data/train/noise_vad_silence'   # 노이즈 파일들
OUTPUT_DIR   = '/home/isol/work/data/train/wakeword_vad_noisy_up'  # 증강 결과 저장

augment_wakeword()

인풋 원본: 7148개
노이즈 파일: 44084개
저장 위치: /home/isol/work/data/train/wakeword_vad_noisy_up
SNR 범위: 1~5 dB
노이즈 파일 로드 중...
노이즈 44084개 로드 완료!

  500/7148 완료...
  1000/7148 완료...
  1500/7148 완료...
  2000/7148 완료...
  2500/7148 완료...
  3000/7148 완료...
  3500/7148 완료...
  4000/7148 완료...
  4500/7148 완료...
  5000/7148 완료...
  5500/7148 완료...
  6000/7148 완료...
  6500/7148 완료...
  7000/7148 완료...

증강 완료!
  성공: 7148개
  실패: 0개
  저장 위치: /home/isol/work/data/train/wakeword_vad_noisy_up

원본 7148개 + 증강 7148개 = 총 14296개


In [36]:
AI_HUB_WAV_DIR   = '/home/isol/work/data/train/AI_Hub/wav'
NOISE_DIR    = '/home/isol/work/data/train/noise_bg_cut'
AI_HUB_NOISY_DIR = '/home/isol/work/data/train/AI_Hub/wav_bg_noise'

augment_aihub()

노이즈 파일 로드 중...
노이즈 21868개 로드 완료!

단어 폴더 수: 280개
폴더당 샘플: 300개
예상 총 출력: 84000개
  N              : 0개 완료
  가람아            : 300개 완료
  검은콩            : 300개 완료
  곰돌아            : 300개 완료
  괜찮아            : 1개 완료
  길동아            : 300개 완료
  길동이            : 1개 완료
  김달콩            : 300개 완료
  까망이            : 300개 완료
  깜냥이            : 300개 완료
  꽁냥이            : 300개 완료
  꽃남아            : 300개 완료
  꽃님아            : 1개 완료
  꿀떡아            : 300개 완료
  꿀순아            : 1개 완료
  끌순아            : 2개 완료
  끝순아            : 300개 완료
  나니몬            : 300개 완료
  눈동이            : 300개 완료
  다나야            : 300개 완료
  다산아            : 300개 완료
  다온아            : 300개 완료
  다은아            : 1개 완료
  다행아            : 300개 완료
  단호박            : 300개 완료
  달맞이            : 300개 완료
  더스티            : 1개 완료
  더스틴            : 300개 완료
  덩어리            : 300개 완료
  데이비드           : 300개 완료
  데이지            : 300개 완료
  도톰아            : 300개 완료
  돌돌아            : 300개 완료
  돌망이            : 1개 완료
  두리야            : 300개

In [37]:
AI_HUB_WAV_DIR   = '/home/isol/work/data/train/AI_Hub/wav'
NOISE_DIR    = '/home/isol/work/data/train/noise_speech_cut'
AI_HUB_NOISY_DIR = '/home/isol/work/data/train/AI_Hub/wav_speech_noise'

augment_aihub()

노이즈 파일 로드 중...
노이즈 32079개 로드 완료!

단어 폴더 수: 280개
폴더당 샘플: 300개
예상 총 출력: 84000개
  N              : 0개 완료
  가람아            : 300개 완료
  검은콩            : 300개 완료
  곰돌아            : 300개 완료
  괜찮아            : 1개 완료
  길동아            : 300개 완료
  길동이            : 1개 완료
  김달콩            : 300개 완료
  까망이            : 300개 완료
  깜냥이            : 300개 완료
  꽁냥이            : 300개 완료
  꽃남아            : 300개 완료
  꽃님아            : 1개 완료
  꿀떡아            : 300개 완료
  꿀순아            : 1개 완료
  끌순아            : 2개 완료
  끝순아            : 300개 완료
  나니몬            : 300개 완료
  눈동이            : 300개 완료
  다나야            : 300개 완료
  다산아            : 300개 완료
  다온아            : 300개 완료
  다은아            : 1개 완료
  다행아            : 300개 완료
  단호박            : 300개 완료
  달맞이            : 300개 완료
  더스티            : 1개 완료
  더스틴            : 300개 완료
  덩어리            : 300개 완료
  데이비드           : 300개 완료
  데이지            : 300개 완료
  도톰아            : 300개 완료
  돌돌아            : 300개 완료
  돌망이            : 1개 완료
  두리야            : 300개

In [38]:
AI_HUB_WAV_DIR   = '/home/isol/work/data/train/AI_Hub/wav'
NOISE_DIR    = '/home/isol/work/data/train/noise_both_cut'
AI_HUB_NOISY_DIR = '/home/isol/work/data/train/AI_Hub/wav_both_noise'

augment_aihub()

노이즈 파일 로드 중...
노이즈 81891개 로드 완료!

단어 폴더 수: 280개
폴더당 샘플: 300개
예상 총 출력: 84000개
  N              : 0개 완료
  가람아            : 300개 완료
  검은콩            : 300개 완료
  곰돌아            : 300개 완료
  괜찮아            : 1개 완료
  길동아            : 300개 완료
  길동이            : 1개 완료
  김달콩            : 300개 완료
  까망이            : 300개 완료
  깜냥이            : 300개 완료
  꽁냥이            : 300개 완료
  꽃남아            : 300개 완료
  꽃님아            : 1개 완료
  꿀떡아            : 300개 완료
  꿀순아            : 1개 완료
  끌순아            : 2개 완료
  끝순아            : 300개 완료
  나니몬            : 300개 완료
  눈동이            : 300개 완료
  다나야            : 300개 완료
  다산아            : 300개 완료
  다온아            : 300개 완료
  다은아            : 1개 완료
  다행아            : 300개 완료
  단호박            : 300개 완료
  달맞이            : 300개 완료
  더스티            : 1개 완료
  더스틴            : 300개 완료
  덩어리            : 300개 완료
  데이비드           : 300개 완료
  데이지            : 300개 완료
  도톰아            : 300개 완료
  돌돌아            : 300개 완료
  돌망이            : 1개 완료
  두리야            : 300개

In [39]:
AI_HUB_WAV_DIR   = '/home/isol/work/data/train/AI_Hub/wav'
NOISE_DIR    = '/home/isol/work/data/train/noise_vad_silence'
AI_HUB_NOISY_DIR = '/home/isol/work/data/train/AI_Hub/wav_vad_noise'

augment_aihub()

노이즈 파일 로드 중...
노이즈 44084개 로드 완료!

단어 폴더 수: 280개
폴더당 샘플: 300개
예상 총 출력: 84000개
  N              : 0개 완료
  가람아            : 300개 완료
  검은콩            : 300개 완료
  곰돌아            : 300개 완료
  괜찮아            : 1개 완료
  길동아            : 300개 완료
  길동이            : 1개 완료
  김달콩            : 300개 완료
  까망이            : 300개 완료
  깜냥이            : 300개 완료
  꽁냥이            : 300개 완료
  꽃남아            : 300개 완료
  꽃님아            : 1개 완료
  꿀떡아            : 300개 완료
  꿀순아            : 1개 완료
  끌순아            : 2개 완료
  끝순아            : 300개 완료
  나니몬            : 300개 완료
  눈동이            : 300개 완료
  다나야            : 300개 완료
  다산아            : 300개 완료
  다온아            : 300개 완료
  다은아            : 1개 완료
  다행아            : 300개 완료
  단호박            : 300개 완료
  달맞이            : 300개 완료
  더스티            : 1개 완료
  더스틴            : 300개 완료
  덩어리            : 300개 완료
  데이비드           : 300개 완료
  데이지            : 300개 완료
  도톰아            : 300개 완료
  돌돌아            : 300개 완료
  돌망이            : 1개 완료
  두리야            : 300개